In [ ]:
import pandas as pd
import json
import uuid
from pathlib import Path

In [ ]:
%load_ext kedro.ipython

In [ ]:
%reload_kedro

In [ ]:
RAW_DATA_DIR = Path("../data/01_raw")

In [ ]:
def process_taxonomy_excel(path: str, sheet_name=0):
    """
    Read taxonomy Excel and automatically compute:
    1. id: unique ID per row
    2. parentCode: code without its last digit (None if code length == 1)
    3. isLeaf: 1 if code length >= 4, else 0

    Parameters
    ----------
    path : str  
        Path to the Excel file.
    sheet_name : str or int  
        Sheet to load (default: first sheet).

    Returns
    -------
    pd.DataFrame : processed taxonomy table.
    """
    categorical_dtypes = {'code': str}

    # Load original Excel
    df = pd.read_excel(path, sheet_name=sheet_name, dtype=categorical_dtypes)

    # 1. Create unique ID for each row (UUID short form)
    df["id"] = [uuid.uuid4().hex[:8] for _ in range(len(df))]


    # 2. Generate parentCode: remove last digit; if length == 1, parent is None
    def compute_parent(code):
        return None if len(code) <= 1 else code[:-1]

    df["parentCode"] = df["code"].apply(compute_parent)

    # 3. Compute isLeaf: 1 if code has >= 4 digits, else 0
    df["isLeaf"] = df["code"].apply(lambda x: 1 if len(x) >= 4 else 0)

    return df

In [ ]:
df_processed = process_taxonomy_excel(RAW_DATA_DIR/"ISCO-08 EN Structure and definitions.xlsx")
df_processed

In [ ]:
df_processed.to_csv(RAW_DATA_DIR/"isco_clean_taxonomy.csv", index=False)

In [ ]:
ISCO_LEVEL_NAMES = {
    "1": "Major group",
    "2": "Sub-major group",
    "3": "Minor group",
    "4": "Unit group",
}


def _clean_text(value):
    if value is None:
        return None
    if isinstance(value, float) and pd.isna(value):
        return None
    text = str(value).strip()
    return text or None


def _clean_code(value):
    text = _clean_text(value)
    if not text:
        return None
    if text.endswith('.0'):
        text = text[:-2]
    return text


nodes_payload = []
for row in df_processed.to_dict('records'):
    nodes_payload.append(
        {
            "code": _clean_code(row.get("code")),
            "level": int(row.get("level", 0)),
            "label": _clean_text(row.get("label")) or "",
            "definition": _clean_text(row.get("definition")),
            "examples": _clean_text(row.get("examples")),
            "parentCode": _clean_code(row.get("parentCode")),
            "isLeaf": bool(int(row.get("isLeaf", 0))),
        }
    )

taxonomy_request = {
    "action": "create",
    "taxonomy": {
        "key": "ISCO",
        "maxDepth": int(df_processed["level"].max()),
        "levelNames": ISCO_LEVEL_NAMES,
        "nodes": nodes_payload,
    },
}

json_path = RAW_DATA_DIR / "taxonomy_request.json"
json_path.write_text(
    json.dumps(taxonomy_request, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
json_path


In [ ]:
taxonomy_dict = catalog.get("taxonomy_request").load()

In [ ]:
taxonomy_payload = taxonomy_dict.get("taxonomy") or {}
taxonomy_payload.get("key")